# Notebook 01: PTB-XL Dataset Loading, Deduplication & Patient-Level Splitting
### MOSAIC-Med Research Study: Investigating Small-Model Multiplicity vs Scale in Healthcare AI

---

### Overview & Objectives
This notebook establishes the clinical testbed for the MOSAIC-Med study using the PTB-XL 12-lead electrocardiography benchmark (Wagner et al., *Scientific Data* 2020; Strodthoff et al., *IEEE JBHI* 2021).
Specifically, we:
1. **Document Dataset Version & Deduplication**:
   - Raw PTB-XL v1.0.1 contained 21,837 records across 18,885 patients.
   - Official PTB-XL v1.0.2 / v1.0.3 changelogs identified and removed **38 duplicate recordings across 16 patients** (`ptbxl/ptbxl_v103_changelog.txt`), yielding exactly **21,799 unique recordings across 18,869 distinct patients**.
2. **Implement Patient-Stratified Splits**:
   - **Training Set (Folds 1–8)**: 17,418 records (15,023 patients)
   - **Validation Set (Fold 9)**: 2,183 records (1,942 patients)
   - **Test Set (Fold 10)**: 2,198 records (1,904 patients)
   - Programmatically verify strict patient separation (zero patient identity leakage).
3. **Verify Signal Preprocessing & Normalization**:
   - Zero-mean unit-variance standardization evaluated per 10-second recording instance across time ($L=1000$) independently per lead:
     $$x'_c(t) = \frac{x_c(t) - \mu_c}{\max(\sigma_c, 10^{-6})}, \quad \text{clipped to } [-10, 10]$$
   - No dataset-level training statistics are used during inference, ensuring total independence.
4. **Clarify Acquisition Site Shift**:
   - Non-primary acquisition sites (`site != 0.0`) within held-out Test Fold 10 serve as an internal domain shift stress test.


In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scratch.py.data import (
    load_ptbxl_metadata, cache_ptbxl_dataset,
    CLASSES, CLASS_NAMES
)
from scratch.py.plotting import set_ieee_style, COLORS

set_ieee_style()
print("Libraries and MOSAIC-Med data modules loaded successfully.")


## 1. Metadata Inspection & Deduplication Verification
Let's inspect the database metadata and verify the 21,799 records and 18,869 unique patients.


In [ ]:
# Load PTB-XL database metadata
df_meta = load_ptbxl_metadata('ptbxl')
print(f"Total Unique Records: {len(df_meta):,}")
print(f"Distinct Patients:    {df_meta['patient_id'].nunique():,}")
print(f"Acquisition Sites:    {df_meta['site'].nunique()} distinct sites")

# Inspect superclass prevalence summary
summary_table = pd.read_csv('results/tables/data_summary.csv')
summary_table


## 2. Patient-Level Stratification Verification (Zero Leakage Check)
To guarantee clinical validity and eliminate optimistic bias, records from the same patient must never cross train, validation, or test partitions.
Let's verify mathematically that there is zero patient overlap.


In [ ]:
train_patients = set(df_meta[df_meta['strat_fold'] <= 8]['patient_id'])
val_patients = set(df_meta[df_meta['strat_fold'] == 9]['patient_id'])
test_patients = set(df_meta[df_meta['strat_fold'] == 10]['patient_id'])

overlap_train_val = len(train_patients.intersection(val_patients))
overlap_train_test = len(train_patients.intersection(test_patients))
overlap_val_test = len(val_patients.intersection(test_patients))

print(f"Train Records: {len(df_meta[df_meta['strat_fold'] <= 8]):,} | Distinct Patients: {len(train_patients):,}")
print(f"Val Records:   {len(df_meta[df_meta['strat_fold'] == 9]):,}  | Distinct Patients: {len(val_patients):,}")
print(f"Test Records:  {len(df_meta[df_meta['strat_fold'] == 10]):,}  | Distinct Patients: {len(test_patients):,}")
print("---")
print(f"Train/Val Patient Overlap:  {overlap_train_val} (Required: 0)")
print(f"Train/Test Patient Overlap: {overlap_train_test} (Required: 0)")
print(f"Val/Test Patient Overlap:   {overlap_val_test} (Required: 0)")

assert overlap_train_val == 0 and overlap_train_test == 0 and overlap_val_test == 0, "Patient leakage detected!"
print(">> Patient-level stratification verified with zero patient overlap across splits.")


## 3. Signal Standardization & Cached Arrays
Raw ECG waveforms are standardized instance-by-instance per lead across the temporal axis ($L=1000$).
Let's inspect preprocessed arrays from disk cache.


In [ ]:
# Load preprocessed arrays from cache
data = cache_ptbxl_dataset('ptbxl', 'scratch/cache')
X_train, y_train = data['X_train'], data['y_train']
X_val, y_val = data['X_val'], data['y_val']
X_test, y_test = data['X_test'], data['y_test']

print(f"X_train shape: {X_train.shape} (float32)")
print(f"X_val shape:   {X_val.shape} (float32)")
print(f"X_test shape:  {X_test.shape} (float32)")

# Plot 12 leads of a representative sample
lead_names = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
fig, axes = plt.subplots(6, 2, figsize=(7.16, 5.0), sharex=True)
sample_idx = 0
time_axis = np.linspace(0, 10, 1000)

for idx, lead in enumerate(lead_names):
    r, c = divmod(idx, 2)
    ax = axes[r, c]
    ax.plot(time_axis, X_train[sample_idx, idx, :], color=COLORS['primary'], linewidth=0.9)
    ax.set_ylabel(lead, fontsize=7.5)
    ax.grid(True, alpha=0.2)
axes[-1, 0].set_xlabel("Time (seconds)")
axes[-1, 1].set_xlabel("Time (seconds)")
fig.suptitle("Preprocessed 12-Lead ECG Sample (100 Hz, 1000 Samples)", fontsize=9.5)
plt.tight_layout()
plt.show()


## 4. Internal Acquisition Site Domain Shift Documentation
> [!NOTE]
> In held-out Test Fold 10 ($N=2,198$), records acquired at non-primary recording sites (`site != 0.0`, $N = 1,711$) exhibit hardware and sensor configuration variations.
> This subset is used as an **internal acquisition site transferability stress test**, but does NOT constitute an independent external hospital cohort.
